# 🎬 MiniMax H3 FreeVideo Studio (Google Colab 1-Click)
### Runs Official `FlashML-org/FreeVideo` MiniMax H3 Engine on Free 16GB T4 GPU

**Features:**
- 100% Free & Unlimited Generations (No API Credits Required!)
- Powered by ComfyUI + FlashML FreeVideo Custom Node
- Generates a Cloudflare Tunnel Public Link so you can control it from your 2GB laptop or phone
- Built-in Auto-Reconnect / Heartbeat to keep Colab alive

In [ ]:
# STEP 1: Verify Hardware Accelerator & Start Heartbeat
import torch
if not torch.cuda.is_available():
    raise SystemError("⚠️ GPU not detected! Go to Runtime -> Change runtime type -> Select 'T4 GPU' and click Save.")

gpu_name = torch.cuda.get_device_name(0)
vram = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1)
print(f"🎮 Active Cloud GPU: {gpu_name} ({vram} GB VRAM)")

# Audio Heartbeat Loop (Prevents Colab from idling/sleeping)
from IPython.display import Audio, display
import numpy as np
silent_audio = np.zeros(44100)
display(Audio(silent_audio, rate=44100, autoplay=True))
print("⚡ Audio heartbeat active (keeps session awake).")

In [ ]:
# STEP 2: Clone ComfyUI & FlashML-org/FreeVideo (MiniMax H3 Engine)
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI

print("📦 Installing ComfyUI core requirements...")
!pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt

print("📥 Cloning FlashML-org/FreeVideo from GitHub...")
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/FlashML-org/FreeVideo.git

%cd /content/ComfyUI/custom_nodes/FreeVideo
print("📦 Installing FreeVideo (MiniMax H3) dependencies...")
!pip install -q -r requirements.txt

%cd /content/ComfyUI
print("✅ ComfyUI + MiniMax FreeVideo Setup Complete!")

In [ ]:
# STEP 3: Launch ComfyUI with Cloudflare Public Tunnel (Open on 2GB Laptop!)
import subprocess
import time
import re
import os

print("🌐 Setting up secure Cloudflare Tunnel...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Start Cloudflare tunnel in background
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for _ in range(40):
    line = tunnel_proc.stdout.readline()
    if "trycloudflare.com" in line:
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if m:
            public_url = m.group(0)
            break
    time.sleep(0.5)

print("\n" + "="*70)
if public_url:
    print(f"🎉 YOUR COMFYUI MINIMAX PUBLIC LINK: {public_url}")
    print("👉 Click the link above to open ComfyUI on your 2GB laptop!")
else:
    print("⚠️ Tunnel starting in background. Check output below for trycloudflare.com URL.")
print("="*70 + "\n")

# Start ComfyUI Server
!python main.py --dont-up-scale-attention --highvram --listen 127.0.0.1 --port 8188